In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json
import re
from datetime import datetime
from groq import Groq
import time
from urllib.parse import urljoin, urlparse
import random

# Initialize Groq client
client = Groq(api_key="")

def llm_query(prompt, model="openai/gpt-oss-120b", max_retries=2):
    """LLM query with minimal retries for speed."""
    for attempt in range(max_retries):
        try:
            chat_completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model,
                temperature=0.1,
                max_tokens=200
            )
            return chat_completion.choices[0].message.content.strip()
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                return ""
    return ""

def extract_khmer_text(text):
    """Fast Khmer text extraction."""
    if not text:
        return ""
    khmer_pattern = re.compile(r'[\u1780-\u17FF\u19E0-\u19FF\s\.\,\!\\?។៕៚៛]+')
    matches = khmer_pattern.findall(text)
    cleaned_text = ' '.join(matches).strip()
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    return cleaned_text

def is_valid_article_url(url, base_domain):
    """Check if URL is likely to be an article page."""
    parsed_url = urlparse(url)
    if base_domain not in parsed_url.netloc:
        return False

    exclude_patterns = [
        r'sitemap', r'feed', r'rss', r'xml', r'wp-json',
        r'category/', r'tag/', r'author/', r'page/',
        r'search', r'comment', r'attachment', r'admin',
        r'\.pdf$', r'\.jpg$', r'\.png$', r'\.zip$',
        r'\?', r'#', r'\d{4}/\d{2}/\d{2}/?$'
    ]

    for pattern in exclude_patterns:
        if re.search(pattern, url, re.IGNORECASE):
            return False

    path_parts = [p for p in parsed_url.path.split('/') if p]
    if len(path_parts) < 2:
        return False

    has_numeric_id = any(re.search(r'\d+', part) for part in path_parts)
    has_content = len(path_parts[-1]) > 10 if path_parts else False

    return has_numeric_id or has_content

def get_smart_delay():
    """Minimal delay for maximum speed."""
    return random.uniform(0.5, 1.5)

def extract_article_content(soup, url):
    """Fast content extraction with multiple fallbacks."""
    content_selectors = [
        '.td-post-content', '.tdb-block-inner', '.entry-content',
        '.post-content', '.article-content', '.content',
        '.story-content', 'article'
    ]

    for selector in content_selectors:
        content_elem = soup.select_one(selector)
        if content_elem:
            for unwanted in content_elem.select('.ads, .advertisement, script, style, iframe'):
                unwanted.decompose()
            content = extract_khmer_text(content_elem.get_text())
            if content and len(content) > 50:
                return content

    all_text = soup.get_text()
    khmer_text = extract_khmer_text(all_text)
    if len(khmer_text) > 100:
        return khmer_text

    return ""

def scrape_single_article_fast(url, headers, source="https://kohsantepheapdaily.com.kh"):
    """Ultra-fast article scraper with minimal validation (JSON structure)."""
    try:
        response = requests.get(url, headers=headers, timeout=8)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.content, 'html.parser')

        # Extract title
        title = ""
        title_selectors = ['h1', '.entry-title', '.post-title', '.article-title', 'title']
        for selector in title_selectors:
            title_elem = soup.select_one(selector)
            if title_elem:
                title = extract_khmer_text(title_elem.get_text())
                if title:
                    break
        if not title:
            return None

        # Extract content
        content = extract_article_content(soup, url)
        if not content or len(content) < 30:
            return None

        # Extract publication date
        date_str = datetime.now().strftime("%m/%d/%Y")
        date_selectors = [
            'time.entry-date', 'time.post-date', '.published',
            'meta[property="article:published_time"]'
        ]
        for selector in date_selectors:
            date_elem = soup.select_one(selector)
            if date_elem:
                if date_elem.get('datetime'):
                    try:
                        date_obj = datetime.fromisoformat(date_elem.get('datetime').split('T')[0])
                        date_str = date_obj.strftime("%m/%d/%Y")
                    except:
                        pass
                elif date_elem.get_text():
                    text_date = date_elem.get_text().strip()
                    if re.search(r'\d{4}-\d{2}-\d{2}', text_date):
                        try:
                            date_obj = datetime.strptime(text_date, "%Y-%m-%d")
                            date_str = date_obj.strftime("%m/%d/%Y")
                        except:
                            pass
                break

        scrape_time = datetime.now().strftime("%m/%d/%Y %H:%M")

        article_data = {
            "url": url,
            "title": title,
            "content": content,
            "source": source,
            "publication_date": date_str,
            "scrape_date": scrape_time
        }

        return article_data

    except Exception:
        return None

def discover_all_urls_aggressive(base_url, headers):
    """Aggressive URL discovery - finds ALL possible articles."""
    base_domain = urlparse(base_url).netloc
    discovered_urls = set()
    processed_pages = set()

    print("Starting aggressive URL discovery...")

    queue = [
        base_url,
        f"{base_url.rstrip('/')}/page/1/",
        f"{base_url.rstrip('/')}/category/national/",
        f"{base_url.rstrip('/')}/category/politics/",
        f"{base_url.rstrip('/')}/category/sport/",
        f"{base_url.rstrip('/')}/category/entertainment/",
        f"{base_url.rstrip('/')}/category/technology/",
        f"{base_url.rstrip('/')}/category/local/",
        f"{base_url.rstrip('/')}/category/life-social/",
        f"{base_url.rstrip('/')}/category/important-news/",
        f"{base_url.rstrip('/')}/category/international/",
        f"{base_url.rstrip('/')}/category/opinion/",
        f"{base_url.rstrip('/')}/category/security/"
    ]


    page_count = 0
    max_pages = 1000

    while queue and page_count < max_pages:
        current_url = queue.pop(0)

        if current_url in processed_pages:
            continue

        try:
            print(f"Scanning: {current_url}")
            response = requests.get(current_url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')

            links = soup.find_all('a', href=True)

            for link in links:
                href = link['href']
                full_url = urljoin(base_url, href)
                if is_valid_article_url(full_url, base_domain):
                    discovered_urls.add(full_url)
                elif (base_domain in urlparse(full_url).netloc and
                      full_url not in processed_pages and
                      full_url not in queue and
                      not any(x in full_url for x in ['sitemap', 'feed', 'xml'])):
                    queue.append(full_url)

            pagination_links = soup.find_all('a', href=True,
                                             string=re.compile(r'[0-9]|next|older|newer|»|›'))
            for pagination_link in pagination_links:
                pagination_url = urljoin(base_url, pagination_link['href'])
                if (base_domain in urlparse(pagination_url).netloc and
                    pagination_url not in processed_pages and
                    pagination_url not in queue):
                    queue.append(pagination_url)

            processed_pages.add(current_url)
            page_count += 1

            print(f"Found {len(discovered_urls)} articles so far...")
            time.sleep(get_smart_delay())

        except Exception as e:
            print(f"Error scanning {current_url}: {e}")
            continue

    url_list = list(discovered_urls)
    print(f"Discovery complete! Found {len(url_list)} potential articles")
    return url_list

def scrape_unlimited(target_url="https://kohsantepheapdaily.com.kh/"):
    """UNLIMITED scraper - gets EVERYTHING it can find."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
    }

    print("STARTING UNLIMITED SCRAPING")
    print("This may take a while...")

    print("\nPHASE 1: URL Discovery")
    all_urls = discover_all_urls_aggressive(target_url, headers)

    if not all_urls:
        print("No URLs found!")
        return []

    print(f"\nPHASE 2: Scraping {len(all_urls)} Articles")
    all_articles = []
    success_count = 0
    fail_count = 0

    for i, url in enumerate(all_urls):
        print(f"[{i+1}/{len(all_urls)}] Processing: {url[:80]}...")
        article_data = scrape_single_article_fast(url, headers)
        if article_data:
            all_articles.append(article_data)
            success_count += 1
            print(f"SUCCESS: {article_data['title'][:60]}...")
        else:
            fail_count += 1
            print("FAILED")

        if (i + 1) % 50 == 0:
            success_rate = (success_count / (i + 1)) * 100
            print(f"Progress: {i+1}/{len(all_urls)} | Success: {success_count} | Failed: {fail_count} | Rate: {success_rate:.1f}%")

        time.sleep(get_smart_delay())

    print("\nUNLIMITED SCRAPING COMPLETE")
    print(f"Total URLs found: {len(all_urls)}")
    print(f"Successfully scraped: {success_count}")
    print(f"Failed: {fail_count}")
    print(f"Success rate: {(success_count/len(all_urls))*100:.1f}%")

    return all_articles

def save_large_dataset(data, filename=None):
    """Save dataset as JSON file."""
    if not data:
        print("No data to save.")
        return None

    if not filename:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"khmer_news_{timestamp}.json"

    unique_data = {item['url']: item for item in data}.values()
    cleaned_data = [item for item in unique_data if len(item.get("content", "")) > 50]

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(cleaned_data, f, ensure_ascii=False, indent=2)

    print(f"SAVED JSON DATASET: {filename}")
    print(f"Total articles saved: {len(cleaned_data)}")
    return filename

def continuous_scraping(target_url="https://dap-news.com/", hours=24):
    """Continuous scraping mode - runs for specified hours."""
    print(f"STARTING CONTINUOUS SCRAPING FOR {hours} HOURS")
    print("Press Ctrl+C to stop early")

    start_time = time.time()
    end_time = start_time + (hours * 3600)
    all_articles = []
    cycle = 1

    try:
        while time.time() < end_time:
            print(f"\nCYCLE {cycle} - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

            new_articles = scrape_unlimited(target_url)
            all_articles.extend(new_articles)

            unique_articles = []
            seen_urls = set()
            for article in all_articles:
                if article['url'] not in seen_urls:
                    unique_articles.append(article)
                    seen_urls.add(article['url'])

            all_articles = unique_articles

            if all_articles:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                progress_file = f"khmer_news_PROGRESS_cycle{cycle}_{timestamp}.json"
                save_large_dataset(all_articles, progress_file)

            print(f"Total unique articles so far: {len(all_articles)}")

            next_cycle_time = time.time() + 3600
            while time.time() < next_cycle_time and time.time() < end_time:
                remaining = min(next_cycle_time - time.time(), end_time - time.time())
                if remaining > 0:
                    print(f"Next cycle in {remaining/60:.1f} minutes...")
                    time.sleep(300)

            cycle += 1

    except KeyboardInterrupt:
        print("Continuous scraping stopped by user")

    print("\nCONTINUOUS SCRAPING FINISHED")
    print(f"Total duration: {(time.time() - start_time)/3600:.1f} hours")
    print(f"Total unique articles collected: {len(all_articles)}")

    if all_articles:
        final_file = save_large_dataset(all_articles, "khmer_news_FINAL_COMPLETE.json")
        return final_file

    return None

if __name__ == "__main__":
    print("KHMER NEWS UNLIMITED SCRAPER")
    print("=" * 50)

    print("MODE 1: One-time unlimited scraping")
    articles = scrape_unlimited("https://kohsantepheapdaily.com.kh/")

    if articles:
        json_path = save_large_dataset(articles)
        with open(json_path, "r", encoding="utf-8") as f:
            sample_data = json.load(f)
            print(f"\nSAMPLE OF {len(sample_data)} ARTICLES:")
            for i, item in enumerate(sample_data[:10]):
                print(f"{i+1}. {item['title'][:80]}...")

    # Uncomment to enable continuous scraping mode
    """
    print("\n" + "="*50)
    print("MODE 2: Continuous scraping (24 hours)")
    continuous_scraping("https://dap-news.com/", hours=24)
    """

KHMER NEWS UNLIMITED SCRAPER
MODE 1: One-time unlimited scraping
STARTING UNLIMITED SCRAPING
This may take a while...

PHASE 1: URL Discovery
Starting aggressive URL discovery...
Scanning: https://kohsantepheapdaily.com.kh/
Found 53 articles so far...
Scanning: https://kohsantepheapdaily.com.kh/page/1/
Found 53 articles so far...
Scanning: https://kohsantepheapdaily.com.kh/category/national/
Found 53 articles so far...
Scanning: https://kohsantepheapdaily.com.kh/category/politics/
Found 53 articles so far...
Scanning: https://kohsantepheapdaily.com.kh/category/sport/
Found 54 articles so far...
Scanning: https://kohsantepheapdaily.com.kh/category/entertainment/
Found 54 articles so far...
Scanning: https://kohsantepheapdaily.com.kh/category/technology/
Found 55 articles so far...
Scanning: https://kohsantepheapdaily.com.kh/category/local/
Found 63 articles so far...
Scanning: https://kohsantepheapdaily.com.kh/category/life-social/
Found 64 articles so far...
Scanning: https://kohsantep

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import json
import re
from datetime import datetime
from groq import Groq
import time
from urllib.parse import urljoin, urlparse
import random

# Initialize Groq client
client = Groq(api_key="")

def llm_query(prompt, model="openai/gpt-oss-120b", max_retries=2):
    """LLM query with minimal retries for speed."""
    for attempt in range(max_retries):
        try:
            chat_completion = client.chat.completions.create(
                messages=[{"role": "user", "content": prompt}],
                model=model,
                temperature=0.1,
                max_tokens=200
            )
            return chat_completion.choices[0].message.content.strip()
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                return ""
    return ""

def extract_khmer_text(text):
    """Fast Khmer text extraction."""
    if not text:
        return ""
    khmer_pattern = re.compile(r'[\u1780-\u17FF\u19E0-\u19FF\s\.\,\!\\?។៕៚៛]+')
    matches = khmer_pattern.findall(text)
    cleaned_text = ' '.join(matches).strip()
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text)
    return cleaned_text

def is_valid_article_url(url, base_domain):
    """Check if URL is likely to be an article page."""
    parsed_url = urlparse(url)
    
    # Must be from same domain
    if base_domain not in parsed_url.netloc:
        return False
    
    # Common non-article patterns to exclude
    exclude_patterns = [
        r'sitemap', r'feed', r'rss', r'xml', r'wp-json',
        r'category/', r'tag/', r'author/', r'page/',
        r'search', r'comment', r'attachment', r'admin',
        r'\.pdf$', r'\.jpg$', r'\.png$', r'\.zip$',
        r'\?', r'#',  # Query strings and anchors
        r'\d{4}/\d{2}/\d{2}/?$'  # Date archive pages
    ]
    
    for pattern in exclude_patterns:
        if re.search(pattern, url, re.IGNORECASE):
            return False
    
    # Should have some path depth
    path_parts = [p for p in parsed_url.path.split('/') if p]
    if len(path_parts) < 2:
        return False
    
    # Should contain numeric ID or meaningful content
    has_numeric_id = any(re.search(r'\d+', part) for part in path_parts)
    has_content = len(path_parts[-1]) > 10 if path_parts else False
    
    return has_numeric_id or has_content

def get_smart_delay():
    """Minimal delay for maximum speed."""
    return random.uniform(0.5, 1.5)

def extract_article_content(soup, url):
    """Fast content extraction with multiple fallbacks."""
    
    # Priority selectors for content
    content_selectors = [
        '.td-post-content', '.tdb-block-inner', '.entry-content',
        '.post-content', '.article-content', '.content',
        '.story-content', 'article'
    ]
    
    for selector in content_selectors:
        content_elem = soup.select_one(selector)
        if content_elem:
            # Quick cleanup
            for unwanted in content_elem.select('.ads, .advertisement, script, style, iframe'):
                unwanted.decompose()
            
            content = extract_khmer_text(content_elem.get_text())
            if content and len(content) > 50:
                return content
    
    # Fallback: get all text and extract Khmer
    all_text = soup.get_text()
    khmer_text = extract_khmer_text(all_text)
    if len(khmer_text) > 100:
        return khmer_text
    
    return ""

def scrape_single_article_fast(url, headers):
    """Ultra-fast article scraper with minimal validation."""
    try:
        response = requests.get(url, headers=headers, timeout=8)
        
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Fast title extraction
        title = ""
        title_selectors = ['h1', '.entry-title', '.post-title', '.article-title', 'title']
        
        for selector in title_selectors:
            title_elem = soup.select_one(selector)
            if title_elem:
                title = extract_khmer_text(title_elem.get_text())
                if title:
                    break
        
        if not title:
            return None
        
        # Fast content extraction
        content = extract_article_content(soup, url)
        if not content or len(content) < 30:
            return None
        
        # Fast date extraction
        date_str = datetime.now().strftime("%Y-%m-%d")
        date_selectors = [
            'time.entry-date', 'time.post-date', '.published',
            'meta[property="article:published_time"]'
        ]
        
        for selector in date_selectors:
            date_elem = soup.select_one(selector)
            if date_elem:
                if date_elem.get('datetime'):
                    date_str = date_elem.get('datetime')[:10]
                break
        
        # Skip LLM summary for speed - we can add it later in batch
        article_data = {
            "Title": title,
            "Date": date_str,
            "Source_URL": url,
            "Content": content,
            "Content_Length": len(content),
            "Scraped_At": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        
        return article_data
        
    except Exception as e:
        return None

def discover_all_urls_aggressive(base_url, headers):
    """Aggressive URL discovery - finds ALL possible articles."""
    base_domain = urlparse(base_url).netloc
    discovered_urls = set()
    processed_pages = set()
    
    print("🔍 Starting aggressive URL discovery...")
    
    # Start with main pages
    queue = [
        base_url,
        f"{base_url.rstrip('/')}/page/1/",
        f"{base_url.rstrip('/')}/category/national/",
        f"{base_url.rstrip('/')}/category/politics/", 
        f"{base_url.rstrip('/')}/category/sport/",
        f"{base_url.rstrip('/')}/category/entertainment/",
        f"{base_url.rstrip('/')}/category/technology/",
    ]
    
    page_count = 0
    max_pages = 1000  # Safety limit to prevent infinite loops
    
    while queue and page_count < max_pages:
        current_url = queue.pop(0)
        
        if current_url in processed_pages:
            continue
            
        try:
            print(f"  📄 Scanning: {current_url}")
            response = requests.get(current_url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Find ALL links on page
            links = soup.find_all('a', href=True)
            
            for link in links:
                href = link['href']
                full_url = urljoin(base_url, href)
                
                # Add valid article URLs
                if is_valid_article_url(full_url, base_domain):
                    discovered_urls.add(full_url)
                
                # Add new pages to queue for crawling
                elif (base_domain in urlparse(full_url).netloc and 
                      full_url not in processed_pages and
                      full_url not in queue and
                      not any(x in full_url for x in ['sitemap', 'feed', 'xml'])):
                    queue.append(full_url)
            
            # Look for pagination
            pagination_links = soup.find_all('a', href=True, 
                                           string=re.compile(r'[0-9]|next|older|newer|»|›'))
            for pagination_link in pagination_links:
                pagination_url = urljoin(base_url, pagination_link['href'])
                if (base_domain in urlparse(pagination_url).netloc and 
                    pagination_url not in processed_pages and
                    pagination_url not in queue):
                    queue.append(pagination_url)
            
            processed_pages.add(current_url)
            page_count += 1
            
            print(f"    ✅ Found {len(discovered_urls)} articles so far...")
            
            time.sleep(get_smart_delay())
            
        except Exception as e:
            print(f"  ⚠️ Error scanning {current_url}: {e}")
            continue
    
    url_list = list(discovered_urls)
    print(f"🎯 Discovery complete! Found {len(url_list)} potential articles")
    return url_list

def scrape_unlimited(target_url="https://dap-news.com/"):
    """
    UNLIMITED scraper - gets EVERYTHING it can find.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.5',
    }
    
    print("🚀 STARTING UNLIMITED SCRAPING - NO LIMITS!")
    print("⚠️ This may take a while...")
    
    # Phase 1: Discover ALL URLs
    print("\n📍 PHASE 1: URL Discovery")
    all_urls = discover_all_urls_aggressive(target_url, headers)
    
    if not all_urls:
        print("❌ No URLs found!")
        return []
    
    # Phase 2: Scrape ALL articles
    print(f"\n📍 PHASE 2: Scraping {len(all_urls)} Articles")
    all_articles = []
    success_count = 0
    fail_count = 0
    
    for i, url in enumerate(all_urls):
        print(f"  [{i+1}/{len(all_urls)}] Processing: {url[:80]}...")
        
        article_data = scrape_single_article_fast(url, headers)
        if article_data:
            all_articles.append(article_data)
            success_count += 1
            print(f"    ✅ SUCCESS: {article_data['Title'][:60]}...")
        else:
            fail_count += 1
            print(f"    ❌ FAILED")
        
        # Progress update every 50 articles
        if (i + 1) % 50 == 0:
            success_rate = (success_count / (i + 1)) * 100
            print(f"📊 Progress: {i+1}/{len(all_urls)} | Success: {success_count} | Failed: {fail_count} | Rate: {success_rate:.1f}%")
        
        time.sleep(get_smart_delay())
    
    # Phase 3: Add summaries in batch (optional)
    if all_articles:
        print(f"\n📍 PHASE 3: Adding Summaries (Optional)")
        for i, article in enumerate(all_articles):
            if i % 10 == 0:  # Add summary to every 10th article to save time
                content_preview = article['Content'][:500]
                summary = llm_query(f"Briefly summarize in English: {content_preview}")
                article['Summary_EN'] = summary
                print(f"  📝 Added summary for article {i+1}")
            else:
                article['Summary_EN'] = ""
    
    print(f"\n🎉 UNLIMITED SCRAPING COMPLETE!")
    print(f"📊 FINAL RESULTS:")
    print(f"   Total URLs found: {len(all_urls)}")
    print(f"   Successfully scraped: {success_count}")
    print(f"   Failed: {fail_count}")
    print(f"   Success rate: {(success_count/len(all_urls))*100:.1f}%")
    
    return all_articles

def save_large_dataset(data, filename=None):
    """Save large datasets with compression options."""
    if not data:
        print("⚠️ No data to save.")
        return None
    
    if not filename:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"khmer_news_COMPLETE_{timestamp}.csv"
    
    df = pd.DataFrame(data)
    
    # Basic cleanup
    df = df.drop_duplicates(subset=['Source_URL'], keep='first')
    df = df[df['Content_Length'] > 50]  # Remove very short articles
    
    # Save to CSV
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    
    print(f"💾 SAVED COMPLETE DATASET: {filename}")
    print(f"📁 Final dataset size: {len(df)} articles")
    print(f"📊 Content statistics:")
    print(f"   - Average content length: {df['Content_Length'].mean():.0f} chars")
    print(f"   - Date range: {df['Date'].min()} to {df['Date'].max()}")
    print(f"   - Total characters: {df['Content_Length'].sum():,}")
    
    return filename

def continuous_scraping(target_url="https://dap-news.com/", hours=24):
    """
    Continuous scraping mode - runs for specified hours.
    """
    print(f"🔄 STARTING CONTINUOUS SCRAPING FOR {hours} HOURS")
    print("Press Ctrl+C to stop early")
    
    start_time = time.time()
    end_time = start_time + (hours * 3600)
    all_articles = []
    cycle = 1
    
    try:
        while time.time() < end_time:
            print(f"\n🔄 CYCLE {cycle} - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            
            # Scrape everything
            new_articles = scrape_unlimited(target_url)
            all_articles.extend(new_articles)
            
            # Remove duplicates
            unique_articles = []
            seen_urls = set()
            for article in all_articles:
                if article['Source_URL'] not in seen_urls:
                    unique_articles.append(article)
                    seen_urls.add(article['Source_URL'])
            
            all_articles = unique_articles
            
            # Save progress
            if all_articles:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                progress_file = f"khmer_news_PROGRESS_cycle{cycle}_{timestamp}.csv"
                save_large_dataset(all_articles, progress_file)
            
            print(f"📈 Total unique articles so far: {len(all_articles)}")
            
            # Wait before next cycle (1 hour)
            next_cycle_time = time.time() + 3600
            while time.time() < next_cycle_time and time.time() < end_time:
                remaining = min(next_cycle_time - time.time(), end_time - time.time())
                if remaining > 0:
                    print(f"⏳ Next cycle in {remaining/60:.1f} minutes...")
                    time.sleep(300)  # Check every 5 minutes
            
            cycle += 1
    
    except KeyboardInterrupt:
        print("\n🛑 Continuous scraping stopped by user")
    
    print(f"\n🎉 CONTINUOUS SCRAPING FINISHED!")
    print(f"🕒 Total duration: {(time.time() - start_time)/3600:.1f} hours")
    print(f"📚 Total unique articles collected: {len(all_articles)}")
    
    # Save final dataset
    if all_articles:
        final_file = save_large_dataset(all_articles, "khmer_news_FINAL_COMPLETE.csv")
        return final_file
    
    return None

# Main execution - CHOOSE YOUR MODE:
if __name__ == "__main__":
    print("🌐 KHMER NEWS UNLIMITED SCRAPER")
    print("=" * 50)
    
    # MODE 1: One-time unlimited scrape
    print("MODE 1: One-time unlimited scraping")
    articles = scrape_unlimited("https://kohsantepheapdaily.com.kh/")
    
    if articles:
        csv_path = save_large_dataset(articles)
        
        # Show sample
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
        print(f"\n📋 SAMPLE OF {len(df)} ARTICLES:")
        for i, (_, row) in enumerate(df.head(10).iterrows()):
            print(f"  {i+1}. {row['Title'][:80]}...")
    
    # UNCOMMENT FOR MODE 2: Continuous scraping (24 hours)
    """
    print("\n" + "="*50)
    print("MODE 2: Continuous scraping (24 hours)")
    continuous_scraping("https://dap-news.com/", hours=24)
    """

🌐 KHMER NEWS UNLIMITED SCRAPER
MODE 1: One-time unlimited scraping
🚀 STARTING UNLIMITED SCRAPING - NO LIMITS!
⚠️ This may take a while...

📍 PHASE 1: URL Discovery
🔍 Starting aggressive URL discovery...
  📄 Scanning: https://kohsantepheapdaily.com.kh/
    ✅ Found 53 articles so far...
  📄 Scanning: https://kohsantepheapdaily.com.kh/page/1/
    ✅ Found 53 articles so far...
  📄 Scanning: https://kohsantepheapdaily.com.kh/category/national/
    ✅ Found 53 articles so far...
  📄 Scanning: https://kohsantepheapdaily.com.kh/category/politics/
    ✅ Found 53 articles so far...
  📄 Scanning: https://kohsantepheapdaily.com.kh/category/sport/
    ✅ Found 54 articles so far...
  📄 Scanning: https://kohsantepheapdaily.com.kh/category/entertainment/
    ✅ Found 54 articles so far...
  📄 Scanning: https://kohsantepheapdaily.com.kh/category/technology/
    ✅ Found 55 articles so far...
  📄 Scanning: https://kohsantepheapdaily.com.kh
    ✅ Found 55 articles so far...
  📄 Scanning: https://kohsantephe